In [0]:
loads = spark.table("workspace.logistics_project.loads")

loads.printSchema()

root
 |-- load_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- load_type: string (nullable = true)
 |-- weight_lbs: long (nullable = true)
 |-- pieces: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- fuel_surcharge: double (nullable = true)
 |-- accessorial_charges: long (nullable = true)
 |-- load_status: string (nullable = true)
 |-- booking_type: string (nullable = true)



In [0]:
from pyspark.sql.functions import month, count, avg, round

seasonal_analysis = (
    loads
    .withColumn("month", month("load_date"))
    .groupBy("month")
    .agg(
        count("load_id").alias("total_loads"),
        round(avg("revenue"), 2).alias("avg_freight_rate")
    )
    .orderBy("month")
)

In [0]:
seasonal_analysis.printSchema()

root
 |-- month: integer (nullable = true)
 |-- total_loads: long (nullable = false)
 |-- avg_freight_rate: double (nullable = true)



In [0]:
display(seasonal_analysis)

month,total_loads,avg_freight_rate
1,7304,3064.16
2,6582,3063.39
3,7213,3100.59
4,7036,3065.49
5,7218,3093.85
6,7106,3109.08
7,7248,3072.19
8,7266,3063.71
9,6970,3052.07
10,7339,3052.93


In [0]:
#Create Gold Schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.logistics_gold
""")

DataFrame[]

In [0]:
#Create Seasonal Analysis DataFrame
from pyspark.sql.functions import month, count, avg, round

loads = spark.table("workspace.logistics_project.loads")

seasonal_analysis = (
    loads
    .withColumn("month", month("load_date"))
    .groupBy("month")
    .agg(
        count("load_id").alias("total_loads"),
        round(avg("revenue"), 2).alias("avg_freight_rate")
    )
    .orderBy("month")
)

display(seasonal_analysis)

month,total_loads,avg_freight_rate
1,7304,3064.16
2,6582,3063.39
3,7213,3100.59
4,7036,3065.49
5,7218,3093.85
6,7106,3109.08
7,7248,3072.19
8,7266,3063.71
9,6970,3052.07
10,7339,3052.93


In [0]:
#Create Gold Table
seasonal_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.logistics_gold.seasonal_patterns")

In [0]:
#MERGE into Gold Table
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.seasonal_patterns"
)

gold_table.alias("target").merge(
    seasonal_analysis.alias("source"),
    """
    target.month = source.month
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#Verify Gold Table
display(
    spark.table("workspace.logistics_gold.seasonal_patterns")
)

month,total_loads,avg_freight_rate
1,7304,3064.16
2,6582,3063.39
3,7213,3100.59
4,7036,3065.49
5,7218,3093.85
6,7106,3109.08
7,7248,3072.19
8,7266,3063.71
9,6970,3052.07
10,7339,3052.93
